# 1. Import Library


In [2]:
import pandas as pd # Untuk manipulasi dan analisis data (seperti membaca file CSV/Excel)
from sklearn.model_selection import train_test_split # Untuk membagi data menjadi data latih (training) dan data uji (testing)
from sklearn.linear_model import LinearRegression # Algoritma inti untuk membuat model prediksi hubungan antar variabel
from sklearn.metrics import r2_score, mean_absolute_error # Untuk mengevaluasi seberapa akurat hasil prediksi model


# 2. Load Dataset

In [6]:
# Membaca data
df = pd.read_csv("MOCA_TEST_ECUADOR.csv", sep=';', encoding='latin1')



# 1. Cek 5 baris pertama untuk melihat isi kolom
print("\n", df.head())

# 2. Cek apakah ada data yang kosong (seperti kasus "Kategori" sebelumnya)
print("\n", df.isnull().sum())

# 3. Cek tipe data tiap kolom (apakah angka atau teks)
print("\n", df.dtypes)



            ID   Edad                      Nivel                  Ocupación   \
0     MOCA0001     50  Universitaria incompleta  Empleado/a tiempo completo   
1     MOCA0002     56  Universitaria incompleta  Empleado/a tiempo completo   
2     MOCA0003     45       Secundaria completa  Empleado/a tiempo completo   
3     MOCA0004     47       Secundaria completa  Empleado/a tiempo completo   
4     MOCA0005     56       Secundaria completa  Empleado/a tiempo completo   
...        ...    ...                       ...                         ...   
1198  MOCA1199     44         primaria completa  Empleado/a tiempo completo   
1199  MOCA1200     86       primaria incompleta                 ama de casa   
1200  MOCA1201     54      universidad completa                 Jubilado/a    
1201  MOCA1202     67    universidad incompleta                 Jubilado/a    
1202  MOCA1203     62      universidad completa                 Jubilado/a    

        Sexo                 Historial Residencia

# 3. DATA PREPROCESSING



*   ID






In [8]:
df = df.drop(columns=["ID "])  # Hapus Kolom Tidak Penting, tambahkan satu spasi di depan


* Edad (Usia)

In [10]:
print(df["Edad "].isnull().sum()) # Cek Missing Value
df["Edad "].fillna(df["Edad "].mean(), inplace=True) # Jika ada isi usia dengan data rata-rata

0


/tmp/ipykernel_10814/2638317276.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Edad "].fillna(df["Edad "].mean(), inplace=True) # Jika ada isi usia dengan data rata-rata


In [12]:
df = df[(df["Edad "] > 40) & (df["Edad "] < 93)] # Sesuaikan dengan konteks dataset (di sini targetnya usia ≥ 20)


* Nivel (Pendidikan)


In [16]:
df["Nivel"] = df["Nivel"].str.lower().str.strip() # Mengubah semua teks menjadi huruf kecil

df["Nivel"] = df["Nivel"].str.replace("promaria", "primaria")
df["Nivel"] = df["Nivel"].str.replace("unversitaria", "universitaria")
# Mengganti huruf typo

def map_nivel(x):
    x = str(x).lower()

    # Tidak sekolah
    if "sin educación" in x:
        return 0

    # SD
    elif "primaria" in x or "primero" in x or "segundo" in x or "tercer" in x or "cuarto" in x:
        return 1

    # SMP/SMA
    elif "secundaria" in x or "bachiller" in x:
        return 2

    # Kuliah / Diploma
    elif ("univers" in x or "superior" in x or
          "tecnolog" in x or "tecnica" in x or "técnica" in x):
        return 3

    # S2/S3
    elif "posgrado" in x or "maestr" in x:
        return 4

    # Noise (hapus / isi nanti)
    elif "estudiante" in x or "profesional" in x:
        return None

    else:
        return None

df["Nivel"] = df["Nivel"].apply(map_nivel)

AttributeError: Can only use .str accessor with string values!

In [13]:
print(df["Nivel"].value_counts()) # Menghitung jumlah kategori pendidikan

Nivel
Tercer Nivel                        153
Secundaria                          134
Primaria                             51
Educación universitaria completa     46
Posgrado                             44
                                   ... 
educación tecnologica                 1
educación universitaria completa      1
educación universtaria completa       1
educación primaria incompleta         1
universidad incompleta                1
Name: count, Length: 100, dtype: int64


In [15]:
df['Nivel'] = df['Nivel'].fillna(0)
df


,Edad,Nivel,Ocupación,Sexo,Historial,Residencia,Alcohol,Tabaco,Actividad Física,Años,Tiempo_minutos_segundos,Cog_num,Cog_ord,Unnamed: 14,Unnamed: 15
0,50,3.0,Empleado/a tiempo completo,Hombre,Nada,Urbano,Nunca,Nunca,Sedentario,11,5:20,26,Desempeño Por Debajo de lo Esperado,NaN,NaN
1,56,3.0,Empleado/a tiempo completo,Hombre,Trastornos psiquiátricos,Urbano,Ocasionalmente,Nunca,Actividad Intensa,11,4:29,23,Desempeño Por Debajo de lo Esperado,NaN,NaN
2,45,2.0,Empleado/a tiempo completo,Mujer,Hipertensión,Rural,Nunca,Nunca,Ligera,18,5:28,24,Desempeño Por Debajo de lo Esperado,NaN,NaN
3,47,2.0,Empleado/a tiempo completo,Hombre,Principio de diabetes,Urbano,Nunca,Nunca,Ligera,8,3:12,10,Desempeño Por Debajo de lo Esperado,NaN,NaN
4,56,2.0,Empleado/a tiempo completo,Hombre,Nada,Urbano,Ocasionalmente,Nunca,Ligera,8,4:16,6,Desempeño Por Debajo de lo Esperado,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,44,1.0,Empleado/a tiempo completo,Mujer,Nada,rural,ocasionalmente,nunca,Actividad Ligera,7,7:49,46,Desempeño Ligeramente Por Debajo de lo Esperado,NaN,NaN
1199,86,1.0,ama de casa,Mujer,Nada,rural,nunca,nunca,sedentario,3,7:37,6,Desempeño Por Debajo de lo Esperado,NaN,NaN
1200,54,3.0,Jubilado/a,Hombre,Nada,urbano,ocasionalmente,fumador opcasional,moderada,15,3:14,47,Desempeño Ligeramente Por Debajo de lo Esperado,NaN,NaN
1201,67,3.0,Jubilado/a,Hombre,Diabetes,urbano,ocasionalmente,ex fumador,Actividad Ligera,15,4:51,12,Desempeño Por Debajo de lo Esperado,NaN,NaN


* Ocupación

In [ ]:
def map_ocupacion(x):
    x = str(x).lower()

    if "desemple" in x or "no especificado" in x:
        return 0
    elif "ama de casa" in x:
        return 1
    elif "jubil" in x:
        return 6
    elif "estudiante" in x:
        return 5
    elif "doctor" in x or "arquitecto" in x or "gerente" in x:
        return 3
    elif "dueño" in x or "emprend" in x:
        return 4
    elif ("empleado" in x or "agric" in x or "obrero" in x or
          "chef" in x or "soldador" in x or "costurera" in x):
        return 2
    else:
        return 2  # default pekerja

df["Ocupación "] = df["Ocupación "].apply(map_ocupacion)

In [ ]:
print(df["Ocupación "].value_counts())

Ocupación 
2    581
0    405
6    101
1     88
5     22
4      3
3      2
Name: count, dtype: int64


* Sexo

In [ ]:
df["Sexo"] = df["Sexo"].str.lower().str.strip()

df = df[df["Sexo"] != "otro"]

df["Sexo"] = df["Sexo"].map({
    "hombre": 1,
    "mujer": 0
})

In [ ]:
print(df["Sexo"].value_counts())

Sexo
1    582
0    575
Name: count, dtype: int64


* Historial
Ubah jadi beberapa fitur penyakit (multi-label)

In [ ]:
df["Historial"] = df["Historial"].str.lower().str.strip()

df["Historial"] = df["Historial"].replace([
    "nada", "ninguno", "ninguna", "ningun", "ninguno"
], "none") # Samakan “tidak ada penyakit”

def cek_penyakit(x, keyword):
    return 1 if keyword in x else 0

df["hipertensi"] = df["Historial"].apply(lambda x: cek_penyakit(x, "hipert"))
df["diabetes"] = df["Historial"].apply(lambda x: cek_penyakit(x, "diabet"))
df["cardio"] = df["Historial"].apply(lambda x: cek_penyakit(x, "card"))
df["neuro"] = df["Historial"].apply(lambda x: cek_penyakit(x, "neuro"))
df["psikiatri"] = df["Historial"].apply(lambda x: cek_penyakit(x, "psiqu"))

In [ ]:
df = df.drop(columns=["Historial", "Unnamed: 14", "Unnamed: 15"]) #Hapue beberapa kolom

* Residencia

In [ ]:
df["Residencia "] = df["Residencia "].str.lower().str.strip()

df["Residencia "] = df["Residencia "].replace({
    "urbana": "urbano",
    "urbanno": "urbano"
})

df["Residencia "] = df["Residencia "].map({
    "urbano": 1,
    "rural": 0
})

In [ ]:
print(df["Residencia "].value_counts())

Residencia 
1.0    832
0.0    324
Name: count, dtype: int64


In [ ]:
df['Residencia '] = df['Residencia '].fillna(0)


* Alcohol

In [ ]:
print(df["Alcohol"].value_counts())

Alcohol
Ocasionalmente     507
Nunca              340
Frecuentemente      67
ocasionalmente      60
nunca               59
Ocasional           39
Frecuente           23
Ocasionalmente      20
Ocacionalmente      14
ocasionalmente       8
Frecuentamente       8
Nada                 3
ocacionalmente       2
frecuente            2
Ocasionalmete        1
Siempre              1
Rara Vez             1
nunxa                1
frecuentemente       1
Name: count, dtype: int64


In [ ]:
df["Alcohol"] = df["Alcohol"].str.lower().str.strip()

df["Alcohol"] = df["Alcohol"].replace({
    "nunxa": "nunca",
    "ocacionalmente": "ocasionalmente",
    "ocasionalmete": "ocasionalmente",
    "frecuentamente": "frecuentemente"
})

def map_alcohol(x):
    if x in ["nunca", "nada"]:
        return 0
    elif "rara" in x:
        return 1
    elif "ocasional" in x:
        return 2
    elif "frecuente" in x:
        return 3
    elif "siempre" in x:
        return 4
    else:
        return None

df["Alcohol"] = df["Alcohol"].apply(map_alcohol)

df["Alcohol"].fillna(df["Alcohol"].mode()[0], inplace=True)

/tmp/ipykernel_15155/4141804679.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Alcohol"].fillna(df["Alcohol"].mode()[0], inplace=True)


* Tabaco


In [ ]:
print(df["Tabaco "].value_counts())

Tabaco 
Nunca                 601
Exfumador              97
Ocasionalmente         92
nunca                  89
Fumador ocasional      50
Ocasional              41
Fumador Ocasional      40
Fumador diario         26
Exfumadora             25
Diario                 18
fumador ocasional      14
Frecuentemente          9
Ocacionalmente          8
ocasionalmente          8
Fumador Diario          7
nunca                   6
ex fumador              4
Ex Fumador              4
exfumador               3
exfumador               2
fumador diario          2
Siempre                 1
Fumadora ocasional      1
ocasional               1
exfumadora              1
nuna                    1
na                      1
Nada                    1
Frecuentamente          1
frecuente               1
ex fumador              1
fumador opcasional      1
Name: count, dtype: int64


In [ ]:
df["Tabaco "] = df["Tabaco "].str.lower().str.strip()

df["Tabaco "] = df["Tabaco "].replace({
    "nuna": "nunca",
    "na": "nunca",
    "ocacionalmente": "ocasionalmente",
    "frecuentamente": "frecuentemente",
    "fumador opcasional": "fumador ocasional"
})

def map_tabaco(x):
    x = str(x)

    if x in ["nunca", "nada"]:
        return 0

    elif "ex" in x:
        return 1

    elif "ocasional" in x:
        return 2

    elif "frecuente" in x:
        return 3

    elif "diario" in x or "siempre" in x:
        return 4

    else:
        return None

df["Tabaco "] = df["Tabaco "].apply(map_tabaco)

df["Tabaco "].fillna(df["Tabaco "].mode()[0], inplace=True)

/tmp/ipykernel_15155/924102402.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Tabaco "].fillna(df["Tabaco "].mode()[0], inplace=True)


* Actividad Física

In [ ]:
print(df["Actividad Física"].value_counts())

Actividad Física
Ligera                                           333
Actividad Ligera                                 194
Moderada                                         175
Sedentario                                       125
Actividad Intensa                                 52
Actividad moderada                                38
Actividad ligera                                  37
Actividad Moderada                                24
No                                                20
Normal                                            19
Intensa                                           17
moderada                                          15
Actividad ligera (caminar, tareas domésticas)     14
Moderado                                          11
Si                                                 9
Sedentario                                         9
Sedentario (poco o ningún ejercicio)               8
Actividad intensa                                  8
Actividad moderada (deporte o

In [ ]:
df["Actividad"] = df["Actividad Física"].str.lower().str.strip()

df["Actividad"] = df["Actividad"].str.replace(r"\(.*\)", "", regex=True)

df["Actividad"] = df["Actividad"].replace({
    "modera": "moderada",
    "sedentaria": "sedentario",
    "moderado": "moderada",
    "intenso": "intensa"
})

def map_actividad(x):
    if "sedent" in x:
        return 0
    elif "ligera" in x:
        return 1
    elif "moder" in x:
        return 2
    elif "intens" in x or "pesado" in x:
        return 3
    elif x in ["si", "no", "normal"]:
        return None  # tidak jelas
    else:
        return None

df["Actividad"] = df["Actividad"].apply(map_actividad)

df["Actividad"].fillna(df["Actividad"].mode()[0], inplace=True)

df = df.drop(columns=["Actividad Física"])

/tmp/ipykernel_15155/967215327.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Actividad"].fillna(df["Actividad"].mode()[0], inplace=True)


In [ ]:
print(df["Actividad"].value_counts())

Actividad
1.0    629
2.0    289
0.0    156
3.0     83
Name: count, dtype: int64


* Años

In [ ]:
print(df["Años"].value_counts())

Años
12    129
18    124
13    105
16     90
15     76
14     71
20     56
6      51
8      44
10     42
17     40
7      39
9      38
19     36
11     33
22     29
5      29
23     19
0      16
21     15
24     10
4       7
25      6
3       6
2       5
48      4
35      4
49      3
56      3
66      2
55      2
42      2
47      2
30      2
54      1
59      1
62      1
65      1
52      1
50      1
68      1
61      1
63      1
57      1
67      1
40      1
27      1
26      1
31      1
29      1
1       1
Name: count, dtype: int64


* Tiempo_minutos_segundos

In [ ]:
def to_seconds(x):
    x = str(x)  # jaga-jaga kalau ada angka

    parts = x.split(':')

    # kasus 1: hanya angka (misalnya "8")
    if len(parts) == 1:
        return int(parts[0]) * 60

    # kasus 2: MM:SS
    elif len(parts) == 2:
        minutes = int(parts[0])
        seconds = int(parts[1])
        return minutes * 60 + seconds

    # kasus 3: HH:MM:SS
    elif len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])
        return hours * 3600 + minutes * 60 + seconds

    else:
        return None


df['Tiempo_seconds'] = df['Tiempo_minutos_segundos'].apply(to_seconds)

In [ ]:
df.drop(columns=['Tiempo_minutos_segundos','Cog_ord'], inplace=True)

In [ ]:
df

,Edad,Nivel,Ocupación,Sexo,Residencia,Alcohol,Tabaco,Años,Cog_num,hipertensi,diabetes,cardio,neuro,psikiatri,Actividad,Tiempo_seconds
0,50,3.0,2,1,1.0,0,0,11,26,0,0,0,0,0,0.0,320
1,56,3.0,2,1,1.0,2,0,11,23,0,0,0,0,1,3.0,269
2,45,2.0,2,0,0.0,0,0,18,24,1,0,0,0,0,1.0,328
3,47,2.0,2,1,1.0,0,0,8,10,0,1,0,0,0,1.0,192
4,56,2.0,2,1,1.0,2,0,8,6,0,0,0,0,0,1.0,256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,44,1.0,2,0,0.0,2,0,7,46,0,0,0,0,0,1.0,469
1199,86,1.0,1,0,0.0,0,0,3,6,0,0,0,0,0,0.0,457
1200,54,3.0,6,1,1.0,2,2,15,47,0,0,0,0,0,2.0,194
1201,67,3.0,6,1,1.0,2,1,15,12,0,1,0,0,0,1.0,291


# MENENTUKAN TARGET DAN FITUR
Kita harus memisahkan data menjadi dua bagian: satu untuk belajar (Training) dan satu untuk ujian (Testing).

* **Fitur :** Edad (umur),Nivel (pendidikan), Ocupación (pekerjaan), Sexo (jenis kelamin), Historial (riwayat penyakit),Residencia (tempat tinggal),Alcohol (alkohol), Tabaco (rokok),
,Actividad (aktivitas fisik) , Años (Umur) ,Tiempo_minutos_segundos (Waktu pengerjaan tes ,format menit:detik)

* **Target** : Cog_num (Skor MoCA : jika pakai regresi) , Cog_ord (Kategori hasil kognitif ; rendah, normal, dll. jika pakai klasifikasi)

In [ ]:

# Jalankan ini dulu agar variabel df_siap tercipta
df_siap = pd.get_dummies(df, drop_first=True)

# Setelah itu baru jalankan pembagian target dan fitur
y = df_siap['Cog_num']
X = df_siap.drop(columns=['Cog_num'])

print("✅ Fitur dan Target telah dipisahkan.")



✅ Fitur dan Target telah dipisahkan.


# MEMBAGI DATA (SPLIT DATA)

In [ ]:
from sklearn.model_selection import train_test_split

# Membagi data menjadi:
# 80% untuk latihan (Training) dan 20% untuk ujian (Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Data latihan: {len(X_train)} baris")
print(f"✅ Data ujian: {len(X_test)} baris")

✅ Data latihan: 925 baris
✅ Data ujian: 232 baris


# MEMBUAT DAN MELATIH MODEL

In [ ]:
from sklearn.linear_model import LinearRegression

# 1. Panggil algoritmanya
model = LinearRegression()

# 2. Proses Belajar (Fit): Komputer mencari pola antara X dan y
model.fit(X_train, y_train)

print("✅ Model selesai dilatih.")

✅ Model selesai dilatih.


# PREDIKSI

In [ ]:
# --- LANGKAH 4: PREDIKSI DAN EVALUASI ---

from sklearn.metrics import mean_absolute_error, r2_score

# Model mencoba menebak data ujian
y_pred = model.predict(X_test)

# Menghitung seberapa akurat model kita
mae = mean_absolute_error(y_test, y_pred) # Rata-rata selisih angka tebakan
r2 = r2_score(y_test, y_pred)            # Skor 0-1 (makin dekat ke 1 makin akurat)

print("\n--- HASIL EVALUASI ---")
print(f"Rata-rata kesalahan (MAE): {mae:.2f}")
print(f"Akurasi Model (R2 Score): {r2:.2f}")


--- HASIL EVALUASI ---
Rata-rata kesalahan (MAE): 21.85
Akurasi Model (R2 Score): 0.11


* MAE (21.85): Artinya, tebakan model Anda rata-rata meleset sebesar 21.85 poin. Jika skor kognitif (Cog_num) aslinya adalah 50, model Anda mungkin menebak 28 atau 72. Untuk skala skor medis, kesalahan sebesar ini biasanya dianggap sangat besar.

* R2 Score (0.11): Artinya, model Anda hanya mampu menjelaskan 11% pola data. Sisanya (89%) adalah kesalahan atau pola yang tidak terbaca oleh model. Nilai ini mendekati 0, yang berarti model Anda nyaris tidak lebih baik daripada sekadar menebak rata-rata.



---

**Mengapa Hasilnya Kurang Bagus?**

Ada beberapa kemungkinan penyebabnya:
* Variabel nivel: Seperti yang Anda sebutkan tadi, jika variabel ini belum diolah dengan benar, model kehilangan informasi penting.
* Hubungan Tidak Linear: Hubungan antara data fisik/medis dengan skor kognitif mungkin tidak bisa dibaca dengan "garis lurus" (LinearRegression).
* Data Kurang: Mungkin fitur (kolom X) yang ada saat ini tidak cukup kuat untuk memprediksi skor kognitif secara akurat.


**Apa yang Harus Dilakukan?**

* Cek Skala Data: Karena MAE-nya besar, pastikan target Cog_num rentang angkanya berapa. Jika rentangnya 0-100, meleset 21 itu besar. Jika rentangnya 0-1000, meleset 21 itu kecil.
* Ganti Algoritma: Cobalah algoritma yang lebih kuat untuk data rumit, seperti Random Forest Regressor.
* Perbaiki Variabel nivel: Pastikan variabel ini sudah berubah menjadi angka dan tidak mengandung data kosong.